# Stochastic flux search with eigenvalue bounds

**What's in this notebook?** This notebook demonstrates `sample_bounded_fluxes`, a stochastic extension of the systematic flux-bounding algorithm of [arXiv:2501.03984](https://arxiv.org/abs/2501.03984). While `enumerate_fluxes` exhaustively enumerates all integer $h$-vectors inside the eigenvalue-derived bounding box, `sample_bounded_fluxes` **randomly samples** $h$-vectors from the box, making it feasible for large $N_{\max}$ or higher $h^{1,2}$ where full enumeration is intractable.

**In this notebook, you will learn:**
- Why `enumerate_fluxes` becomes infeasible at large $N_{\max}$ (exponential cost) and how `sample_bounded_fluxes` avoids this
- How to configure `sample_bounded_fluxes`: `n_target`, `n_batch`, `n_mod`, and `max_batches`
- The difference between `refine=False` (ISD-completed candidate fluxes) and `refine=True` (actual $D_I W = 0$ vacua)
- How to scale to large $N_{\max} = 200$ where the bounding box contains $\sim 10^{12}$ candidates

**Prerequisites:** Eigenvalue flux bounding ([notebook 10](10_flux_bounding.ipynb)).

**Related notebooks:** [10_flux_bounding](10_flux_bounding.ipynb) (systematic enumeration, same infrastructure), [10c_sample_bounded_fluxes_stepbystep](10c_sample_bounded_fluxes_stepbystep.ipynb) (step-by-step algorithm walkthrough), [9_sampling_vacua_from_fluxes](../02_vacuum_finding/9_sampling_vacua_from_fluxes.ipynb) (Newton refinement from a given flux pool).

(*Created:* Andreas Schachner, 2025)

## Motivation

The systematic enumeration algorithm (`enumerate_fluxes`) enumerates **all** integer NSNS-flux vectors $h = (h_1, h_2)$ inside the eigenvalue-derived bounding box. The number of candidates scales as

$$
    N_{\text{candidates}} \sim \left(\frac{N_{\max}}{s_{\min}\,\mu_{\min}}\right)^{h^{1,2}+1},
$$

which grows polynomially in $N_{\max}$ and exponentially in $h^{1,2}$. For the $h^{1,2} = 2$ model at $N_{\max} = 20$ this is manageable ($\sim 10^4$ candidates), but at $N_{\max} = 200$ or for $h^{1,2} \geq 3$ the box can contain $10^8$+ candidates.

`sample_bounded_fluxes` replaces the exhaustive enumeration with **uniform random sampling** inside the same bounding ellipsoid. The eigenvalue bounds still focus the search into the proven-valid region of flux space, but the computational cost is $O(n_{\text{target}} / \text{yield})$ — independent of $N_{\max}$.

## Imports

In [ ]:
# General imports
import warnings
import numpy as np

# JAX imports
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import matplotlib.pyplot as plt

# JAXVacua
import jaxvacua as jvc
from jaxvacua.flux_bounding import bounded_fluxes

warnings.filterwarnings('ignore')

## Model and sampler setup

We use the same $h^{1,2} = 2$ Kreuzer–Skarke geometry as in notebook 10 (flux bounding).

In [ ]:
h12 = 2
model = jvc.FluxVacuaFinder(
    h12=h12,
    model_ID=1,
    maximum_degree=2,
    model_type="KS",
)

model

In [ ]:
sampler = jvc.data_sampler(
    model,
    moduli_bounds=(2., 3.),
    axion_bounds=(-0.5, 0.5),
    dilaton_bounds=(2., 10.),
)
sampler

## 1. Stochastic search at moderate $N_{\max}$

At $N_{\max} = 20$, both `enumerate_fluxes` and `sample_bounded_fluxes` are feasible. This lets us validate the stochastic method against the systematic one.

In [ ]:
bf = bounded_fluxes(model, sampler=sampler, Nmax=10)
bf.compute_eigenvalue_bounds(n_sample=10_000)
print(bf)

### Systematic enumeration (baseline)

In [ ]:
try:
    enum_results = bf.enumerate_fluxes(
        n_sample=300,
        max_h_candidates=2_000_000,
        n_isd_per_h=30,
        verbose=True,
        return_moduli=True,
    )
    print(f"\nEnumeration found {len(enum_results)} valid flux candidates.")
except RuntimeError as e:
    print(f"\n[Note] Enumeration infeasible with this model/Nmax: {e}")
    print("Setting enum_results = [] — downstream comparison cells will be skipped.")
    enum_results = []

In [ ]:
try:
    enum_results = bf.enumerate_fluxes(
        n_sample=1_000,
        max_h_candidates=5_000_000,
        n_isd_per_h=100,
        verbose=True,
        return_moduli=True,
        refine=True,
    )
    print(f"\nEnumeration found {len(enum_results)} valid flux candidates.")
except RuntimeError as e:
    print(f"\n[Note] Enumeration infeasible with this model/Nmax: {e}")
    print("Setting enum_results = [] — downstream comparison cells will be skipped.")
    enum_results = []

### Stochastic search

The key parameters are:
- `n_target`: stop after finding this many unique valid flux candidates.
- `n_batch`: number of random $h$-vectors sampled per batch (controls memory vs. overhead).
- `n_mod`: number of moduli points for ISD completion (same role as `n_isd_per_h`).
- `max_batches`: upper limit on sampling rounds.
- `seed`: for reproducibility of the $h$-sampling.

**New parameters:**
- `moduli_regions`: stratified moduli starting points (Cartesian product across all $h^{1,2}$ dimensions).
- `constraints`: user-supplied constraint function `(moduli, tau, flux) → bool`.
- `use_linearised_shifts`: iterative ISD refinement via `linearised_shifts` with flag-based early stopping.
- `chunk_size`: override the default streaming chunk size.

In [ ]:
bf_stoch = bounded_fluxes(model, sampler=sampler, Nmax=20)
bf_stoch.compute_eigenvalue_bounds(n_sample=10_000)

stoch_results = bf_stoch.sample_bounded_fluxes(
    n_target=1_000,
    n_batch=50_000,
    n_sample=1_000,
    n_mod=20,
    max_batches=50,
    refine=True,
    newton_step_size=1.,
    verbose=True,
    seed=42,
    return_moduli=True,
    moduli_regions=[(2., 2.5), (2.5, 3.)],
)
print(f"\nStochastic search found {len(stoch_results)} unique valid flux candidates.")

In [ ]:
# Each result is a dict with "flux", "moduli", "tau"
if stoch_results:
    r = stoch_results[0]
    print("flux   =", r["flux"].astype(int))
    print("moduli =", r["moduli"])
    print("tau    =", r["tau"])
    z = r["moduli"]
    t = r["tau"]
    flux = r["flux"].astype(int)
    DW = model.DW(z,jnp.conj(z),t,jnp.conj(t),flux)
    print("DW    =", DW)

### Comparison

We check how many of the stochastic results also appear in the exhaustive enumeration.

In [ ]:
# Build set of flux tuples from enumeration
enum_set = set()
for r in enum_results:
    enum_set.add(tuple(int(x) for x in r["flux"]))

# Check overlap
n_overlap = 0
for r in stoch_results:
    key = tuple(int(x) for x in r["flux"])
    if key in enum_set:
        n_overlap += 1

print(f"Enumeration:  {len(enum_results)} flux candidates")
print(f"Stochastic:   {len(stoch_results)} flux candidates")
print(f"Overlap:      {n_overlap} / {len(stoch_results)} stochastic results also found by enumeration")
if len(enum_results) > 0:
    print(f"Coverage:     {n_overlap}/{len(enum_results)} = {100*n_overlap/len(enum_results):.1f}% of enumerated vacua recovered")

### Tadpole distributions

In [ ]:
def compute_tadpoles(bf_obj, results):
    return np.array([bf_obj.get_nflux(jnp.array(r["flux"])) for r in results])

if len(enum_results) > 0 and len(stoch_results) > 0:
    tad_enum  = compute_tadpoles(bf, enum_results)
    tad_stoch = compute_tadpoles(bf_stoch, stoch_results)

    fig, axes = plt.subplots(1, 3, dpi=150, figsize=(13, 3.5))

    bins = range(0, int(max(tad_enum.max(), tad_stoch.max())) + 2)

    axes[0].hist(tad_enum, bins=bins, edgecolor='k', linewidth=0.5, alpha=0.7, label='Enumeration')
    axes[0].hist(tad_stoch, bins=bins, edgecolor='k', linewidth=0.5, alpha=0.5, label='Stochastic')
    axes[0].set_xlabel(r"$N_{\rm flux}$", fontsize=12)
    axes[0].set_ylabel("Count", fontsize=12)
    axes[0].set_title("D3-tadpole distribution", fontsize=11)
    axes[0].legend(fontsize=9)

    # h-norm distributions
    def h_norms(bf_obj, results):
        return np.array([bf_obj.compute_norm(bf_obj.get_fh(jnp.array(r["flux"]))[1]) for r in results])

    hn_enum  = h_norms(bf, enum_results)
    hn_stoch = h_norms(bf_stoch, stoch_results)

    axes[1].hist(hn_enum, bins=20, edgecolor='k', linewidth=0.5, alpha=0.7, label='Enumeration')
    axes[1].hist(hn_stoch, bins=20, edgecolor='k', linewidth=0.5, alpha=0.5, label='Stochastic')
    axes[1].set_xlabel(r"$\|h\|^2$", fontsize=12)
    axes[1].set_ylabel("Count", fontsize=12)
    axes[1].set_title(r"NSNS-flux norm distribution", fontsize=11)
    axes[1].legend(fontsize=9)

    # Moduli scatter: Im(z_1) vs Im(z_2) at which each stochastic flux was found valid
    mod_arr = np.array([r["moduli"] for r in stoch_results])
    sc = axes[2].scatter(mod_arr[:, 0].imag, mod_arr[:, 1].imag,
                         c=tad_stoch, cmap="viridis", s=15, alpha=0.7)
    plt.colorbar(sc, ax=axes[2], label=r"$N_{\rm flux}$")
    axes[2].set_xlabel(r"$\mathrm{Im}(z_1)$", fontsize=12)
    axes[2].set_ylabel(r"$\mathrm{Im}(z_2)$", fontsize=12)
    axes[2].set_title("Moduli of stochastic vacua", fontsize=11)

    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to plot.")

## 2. Scaling to large $N_{\max}$

At $N_{\max} = 200$, the bounding box contains far too many $h$-vectors for exhaustive enumeration. But `sample_bounded_fluxes` handles this without difficulty.

In [ ]:
bf_large = bounded_fluxes(model, sampler=sampler, Nmax=200)
bf_large.compute_eigenvalue_bounds(n_sample=10_000)

dim = bf_large.dimension_H3
h1_box, h2_box, h_box = bf_large._h1_box, bf_large._h2_box, bf_large._h_box
h1_max = int(np.ceil(h1_box))
h2_max = int(np.ceil(h2_box))
n_box = (2 * h1_max + 1) ** dim * (2 * h2_max + 1) ** dim

print(f"Nmax = {bf_large.Nmax}")
print(f"Bounding box: h1_box={h1_box:.2f}, h2_box={h2_box:.2f}, h_box={h_box:.2f}")
print(f"Unfiltered box size: {n_box:,} h-candidates")
print(f"  → Full enumeration is {'feasible' if n_box < 1_000_000 else 'INFEASIBLE'}")

In [ ]:
%%time
large_results = bf_large.sample_bounded_fluxes(
    n_target=500,
    n_batch=100_000,
    n_sample=300,
    n_mod=15,
    max_batches=50,
    verbose=True,
    seed=123,
    return_moduli=True,
)
print(f"\nFound {len(large_results)} unique flux candidates at Nmax={bf_large.Nmax}.")

In [ ]:
if len(large_results) > 0:
    tad_large = compute_tadpoles(bf_large, large_results)
    mod_large = np.array([r["moduli"] for r in large_results])

    fig, axes = plt.subplots(1, 2, dpi=150, figsize=(9, 3.5))

    axes[0].hist(tad_large, bins=30, edgecolor='k', linewidth=0.5)
    axes[0].set_xlabel(r"$N_{\rm flux}$", fontsize=12)
    axes[0].set_ylabel("Count", fontsize=12)
    axes[0].set_title(f"Tadpole distribution ($N_{{\\max}} = {bf_large.Nmax}$)", fontsize=11)

    sc = axes[1].scatter(mod_large[:, 0].imag, mod_large[:, 1].imag,
                         c=tad_large, cmap="viridis", s=15, alpha=0.7)
    plt.colorbar(sc, ax=axes[1], label=r"$N_{\rm flux}$")
    axes[1].set_xlabel(r"$\mathrm{Im}(z_1)$", fontsize=12)
    axes[1].set_ylabel(r"$\mathrm{Im}(z_2)$", fontsize=12)
    axes[1].set_title("Moduli locations", fontsize=11)

    plt.tight_layout()
    plt.show()

    print(f"N_flux range: [{tad_large.min():.0f}, {tad_large.max():.0f}]")
    print(f"N_flux mean:  {tad_large.mean():.1f}")
else:
    print("No flux candidates found.")

## 3. Newton refinement

With `refine=True`, each candidate flux is Newton-refined to solve $D_I W = 0$ exactly. The result is filtered by convergence and patch membership, then deduplicated.

**Why `refine=True` may yield 0 converged vacua:** The ISD completion uses `n_mod` randomly sampled moduli points as Newton starting guesses for each integer-rounded flux vector. The actual SUSY vacuum for a given flux typically lies at a moduli location *different* from these `n_mod` sampled points. With small `n_mod` (e.g. `n_mod=15`), coverage is too sparse and Newton diverges for essentially all candidates. Practical guidance:

- Use **`n_mod ≥ 100`** (ideally 300–500) to have a reasonable chance that at least one starting point is close to the vacuum.
- Keep **`newton_step_size=1.0`** (the default) — this gives quadratic convergence when the starting point is close to a root. Reducing the step size does not enlarge the basin of attraction and only slows convergence.
- For large-scale searches, the recommended workflow is to run `refine=False` first to collect a pool of candidate fluxes, then refine them with the doubly-vmapped optimiser from [notebook 9](../02_vacuum_finding/9_sampling_vacua_from_fluxes.ipynb), which uses a much larger and better-distributed set of moduli starting points.

The example below illustrates the 0-convergence case with `n_mod=15`; the output is the expected behaviour, not a bug.

In [ ]:
bf_refine = bounded_fluxes(model, sampler=sampler, Nmax=20)
bf_refine.compute_eigenvalue_bounds(n_sample=10_000)

refined = bf_refine.sample_bounded_fluxes(
    n_target=1000,
    n_batch=500000,
    n_sample=300,
    n_mod=15,
    max_batches=50,
    verbose=True,
    seed=42,
    refine=True,
    newton_tol=1e-10,
    newton_max_iters=100,
)

In [ ]:
if len(refined) > 0:
    print(f"Found {len(refined)} refined SUSY vacua.\n")
    print(f"{'Flux':<45} {'|residual|':>12} {'Im(tau)':>10}")
    print("-" * 70)
    for r in refined[:10]:
        flux_str = str(r['flux'].astype(int).tolist())
        print(f"{flux_str:<45} {r['residual']:12.2e} {r['tau'].imag:10.4f}")
    if len(refined) > 10:
        print(f"  ... ({len(refined) - 10} more)")
else:
    print("No refined vacua found — try increasing n_target or loosening newton_tol.")

## Summary

| Method | Use case | Completeness | Cost scaling |
|--------|----------|-------------|---------------|
| `enumerate_fluxes` | Small $N_{\max}$, low $h^{1,2}$ | Provably complete | $O(N_{\max}^{h^{1,2}+1})$ |
| `sample_bounded_fluxes` | Large $N_{\max}$, any $h^{1,2}$ | Probabilistic | $O(n_{\text{target}} / \text{yield})$ |

Both methods share the same infrastructure:
- Global eigenvalue bounds from a moduli sample (cached via `compute_eigenvalue_bounds`).
- The same JIT-compiled ISD completion + bound-checking kernel.
- Optional Newton refinement and patch membership filtering.
- `KeyboardInterrupt` gracefully returns partial results found so far.

**New features:**
- `compute_eigenvalue_bounds(n_sample)` — pre-compute once, reuse across calls.
- `moduli_regions` — stratified Im(z) bands (Cartesian product across moduli dimensions).
- `use_linearised_shifts` — iterative ISD refinement with flag-based early stopping.
- `constraints` — user-supplied constraint function.
- `chunk_size` — override streaming chunk size.

The stochastic method uses the eigenvalue-derived bounding ellipsoid to concentrate sampling into the region of flux space where valid vacua exist, making it significantly more efficient than blind random sampling.